In [14]:
# Cell 1. EfficientAT clone + 최소 의존성 설치
import os
import sys
import subprocess

REPO_DIR = "/content/EfficientAT"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/fschmid56/EfficientAT.git", REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
print("Current directory:", os.getcwd())

# Colab의 torch/torchvision/torchaudio는 그대로 사용하고,
# inference에 필요한 오디오/표시 패키지만 설치
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "librosa", "soundfile", "resampy", "pandas", "tqdm"
    ],
    check=True
)

# mp3/wav 로딩용 ffmpeg
subprocess.run(
    ["bash", "-lc", "apt-get -qq update && apt-get -qq install -y ffmpeg"],
    check=True
)

print("Setup done.")

Current directory: /content/EfficientAT
Setup done.


In [15]:
# Cell 2. EfficientAT 공식 inference.py 실행
import os
import sys
import torch
import subprocess

os.chdir("/content/EfficientAT")

MODEL_NAME = "mn10_as"
AUDIO_PATH = "resources/metro_station-paris.wav"

cmd = [
    sys.executable,
    "inference.py",
    f"--model_name={MODEL_NAME}",
    f"--audio_path={AUDIO_PATH}",
]

if torch.cuda.is_available():
    cmd.insert(2, "--cuda")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

Running: /usr/bin/python3 inference.py --model_name=mn10_as --audio_path=resources/metro_station-paris.wav


CompletedProcess(args=['/usr/bin/python3', 'inference.py', '--model_name=mn10_as', '--audio_path=resources/metro_station-paris.wav'], returncode=0)

In [16]:
# Cell 3. PC에서 wav/mp3 업로드
from google.colab import files

uploaded = files.upload()
AUDIO_PATH = next(iter(uploaded.keys()))

print("Uploaded audio:", AUDIO_PATH)

Saving 9946DF375D0D0A143B.mp3 to 9946DF375D0D0A143B (1).mp3
Uploaded audio: 9946DF375D0D0A143B (1).mp3


In [17]:
# Cell 4. 업로드 파일을 공식 inference.py로 실행
import sys
import torch
import subprocess

MODEL_NAME = "mn10_as"

cmd = [
    sys.executable,
    "inference.py",
    f"--model_name={MODEL_NAME}",
    f"--audio_path={AUDIO_PATH}",
]

if torch.cuda.is_available():
    cmd.insert(2, "--cuda")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

Running: /usr/bin/python3 inference.py --model_name=mn10_as --audio_path=9946DF375D0D0A143B (1).mp3


CompletedProcess(args=['/usr/bin/python3', 'inference.py', '--model_name=mn10_as', '--audio_path=9946DF375D0D0A143B (1).mp3'], returncode=0)

In [18]:
# Cell 5. EfficientAT Runner 정의
import os
import sys
import io
import math
import time
import contextlib
import numpy as np
import pandas as pd
import torch
import librosa

os.chdir("/content/EfficientAT")
sys.path.insert(0, "/content/EfficientAT")

from models.mn.model import get_model as get_mn
from models.dymn.model import get_model as get_dymn
from models.preprocess import AugmentMelSTFT
from helpers.utils import NAME_TO_WIDTH, labels


class EfficientATRunner:
    def __init__(
        self,
        model_name: str = "mn10_as",
        sample_rate: int = 32000,
        n_mels: int = 128,
        win_length: int = 800,
        hop_size: int = 320,
        device: str | None = None,
    ):
        self.model_name = model_name
        self.sample_rate = sample_rate
        self.n_mels = n_mels
        self.win_length = win_length
        self.hop_size = hop_size
        self.device = torch.device(
            device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )

        # fc head 모델을 쓸 때만 head_type을 바꿔줌
        head_type = "fully_convolutional" if "_fc" in model_name else "mlp"

        # get_model이 모델 구조를 길게 print하므로 출력 억제
        with contextlib.redirect_stdout(io.StringIO()):
            if model_name.startswith("dymn"):
                self.model = get_dymn(
                    width_mult=NAME_TO_WIDTH(model_name),
                    pretrained_name=model_name,
                )
            else:
                self.model = get_mn(
                    width_mult=NAME_TO_WIDTH(model_name),
                    pretrained_name=model_name,
                    head_type=head_type,
                )

        self.model.to(self.device)
        self.model.eval()

        self.mel = AugmentMelSTFT(
            n_mels=n_mels,
            sr=sample_rate,
            win_length=win_length,
            hopsize=hop_size,
        ).to(self.device)
        self.mel.eval()

        print(f"Loaded EfficientAT: {model_name}")
        print(f"Device: {self.device}")

    def _predict_probs_from_waveform(self, waveform_np: np.ndarray) -> np.ndarray:
        if waveform_np.ndim != 1:
            raise ValueError("waveform_np must be mono 1D audio.")

        waveform = torch.from_numpy(waveform_np[None, :]).float().to(self.device)

        with torch.inference_mode():
            if self.device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    spec = self.mel(waveform)
                    logits, features = self.model(spec.unsqueeze(0))
            else:
                spec = self.mel(waveform)
                logits, features = self.model(spec.unsqueeze(0))

        probs = torch.sigmoid(logits.float()).squeeze().detach().cpu().numpy()
        return probs

    def topk_df(self, probs: np.ndarray, top_k: int = 10) -> pd.DataFrame:
        idx = np.argsort(probs)[::-1][:top_k]
        return pd.DataFrame({
            "rank": np.arange(1, len(idx) + 1),
            "audioset_label": [labels[i] for i in idx],
            "probability": [float(probs[i]) for i in idx],
        })

    def predict_file(self, audio_path: str, top_k: int = 10, return_probs: bool = False):
        waveform_np, _ = librosa.load(
            audio_path,
            sr=self.sample_rate,
            mono=True,
        )

        start = time.perf_counter()
        probs = self._predict_probs_from_waveform(waveform_np)
        elapsed = time.perf_counter() - start

        df = self.topk_df(probs, top_k=top_k)
        df["audio_path"] = audio_path
        df["inference_sec"] = elapsed

        if return_probs:
            return df, probs
        return df

    def predict_windows(
        self,
        audio_path: str,
        window_sec: float = 3.0,
        hop_sec: float = 1.5,
        top_k: int = 5,
    ) -> pd.DataFrame:
        waveform_np, _ = librosa.load(
            audio_path,
            sr=self.sample_rate,
            mono=True,
        )

        win = int(window_sec * self.sample_rate)
        hop = int(hop_sec * self.sample_rate)
        n = len(waveform_np)

        if n == 0:
            raise ValueError("Audio file is empty.")

        n_windows = max(1, math.ceil(max(0, n - win) / hop) + 1)

        records = []
        for w in range(n_windows):
            s = w * hop
            e = s + win

            segment = waveform_np[s:min(e, n)]

            if len(segment) < win:
                segment = np.pad(segment, (0, win - len(segment)))

            probs = self._predict_probs_from_waveform(segment)
            top_df = self.topk_df(probs, top_k=top_k)

            for _, row in top_df.iterrows():
                records.append({
                    "window_id": w,
                    "start_sec": s / self.sample_rate,
                    "end_sec": min(e, n) / self.sample_rate,
                    "rank": int(row["rank"]),
                    "audioset_label": row["audioset_label"],
                    "probability": float(row["probability"]),
                })

        return pd.DataFrame(records)

In [19]:
# Cell 6. 업로드 오디오 top-10 추론
MODEL_NAME = "mn10_as"

runner = EfficientATRunner(model_name=MODEL_NAME)

top_df, probs = runner.predict_file(
    AUDIO_PATH,
    top_k=10,
    return_probs=True,
)

display(top_df)

/usr/local/lib/python3.12/dist-packages/torchvision/ops/misc.py:121: UserWarning: Don't use ConvNormActivation directly, please use Conv2dNormActivation and Conv3dNormActivation instead.
  warnings.warn(


Loaded EfficientAT: mn10_as
Device: cpu


/content/EfficientAT/models/preprocess.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


,rank,audioset_label,probability,audio_path,inference_sec
0,1,"Whoosh, swoosh, swish",0.245846,9946DF375D0D0A143B (1).mp3,0.185194
1,2,Field recording,0.204311,9946DF375D0D0A143B (1).mp3,0.185194
2,3,Vehicle,0.183513,9946DF375D0D0A143B (1).mp3,0.185194
3,4,Rustle,0.080817,9946DF375D0D0A143B (1).mp3,0.185194
4,5,Aircraft,0.068428,9946DF375D0D0A143B (1).mp3,0.185194
5,6,Sound effect,0.058444,9946DF375D0D0A143B (1).mp3,0.185194
6,7,White noise,0.053720,9946DF375D0D0A143B (1).mp3,0.185194
7,8,Silence,0.050853,9946DF375D0D0A143B (1).mp3,0.185194
8,9,"Outside, urban or manmade",0.047037,9946DF375D0D0A143B (1).mp3,0.185194
9,10,Car,0.045560,9946DF375D0D0A143B (1).mp3,0.185194


In [20]:
# Cell 7. 프로젝트 관심 라벨 필터링
TARGET_KEYWORDS = {
    "차량 경적": ["vehicle horn", "car horn", "honking", "horn"],
    "사이렌": ["siren", "emergency vehicle"],
    "비명/고함": ["screaming", "scream", "yell", "shout"],
    "화재/경보": ["fire alarm", "smoke detector", "smoke alarm", "alarm"],
    "초인종": ["doorbell"],
    "전화벨": ["telephone bell", "ringtone", "ringing"],
    "아기 울음": ["baby cry", "infant cry", "crying"],
    "개 짖음": ["dog bark", "bark"],
    "물/누수 후보": ["water", "drip", "dripping", "splash", "pour"],
}


def target_scores_from_probs(
    probs: np.ndarray,
    target_keywords: dict[str, list[str]] = TARGET_KEYWORDS,
) -> pd.DataFrame:
    label_lower = [x.lower() for x in labels]

    rows = []
    for target_name, keywords in target_keywords.items():
        keywords_lower = [k.lower() for k in keywords]

        matched_idx = [
            i for i, label in enumerate(label_lower)
            if any(k in label for k in keywords_lower)
        ]

        if not matched_idx:
            continue

        best_idx = max(matched_idx, key=lambda i: probs[i])

        rows.append({
            "target_name_ko": target_name,
            "best_audioset_label": labels[best_idx],
            "probability": float(probs[best_idx]),
            "matched_label_count": len(matched_idx),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("probability", ascending=False)
        .reset_index(drop=True)
    )


target_df = target_scores_from_probs(probs)
display(target_df)

,target_name_ko,best_audioset_label,probability,matched_label_count
0,물/누수 후보,Water,0.002963,7
1,차량 경적,"Vehicle horn, car horn, honking",0.000610,5
2,전화벨,Ringtone,0.000529,3
3,사이렌,Siren,0.000482,6
4,개 짖음,Bark,0.000316,1
5,비명/고함,Shout,0.000270,4
6,화재/경보,Alarm,0.000124,5
7,아기 울음,"Crying, sobbing",0.000115,2
8,초인종,Doorbell,0.000009,1


In [21]:
# Cell 8. 임시 위험도 후처리
DANGER_TARGETS = {"차량 경적", "사이렌", "비명/고함", "화재/경보"}
INFO_TARGETS = {"초인종", "전화벨", "아기 울음", "개 짖음", "물/누수 후보"}


def classify_alert(row, threshold_danger=0.25, threshold_info=0.20):
    name = row["target_name_ko"]
    p = row["probability"]

    if name in DANGER_TARGETS and p >= threshold_danger:
        return "위험"
    if name in INFO_TARGETS and p >= threshold_info:
        return "주의/정보"
    return "낮음"


alert_df = target_df.copy()
alert_df["alert_level"] = alert_df.apply(classify_alert, axis=1)

display(alert_df)

,target_name_ko,best_audioset_label,probability,matched_label_count,alert_level
0,물/누수 후보,Water,0.002963,7,낮음
1,차량 경적,"Vehicle horn, car horn, honking",0.000610,5,낮음
2,전화벨,Ringtone,0.000529,3,낮음
3,사이렌,Siren,0.000482,6,낮음
4,개 짖음,Bark,0.000316,1,낮음
5,비명/고함,Shout,0.000270,4,낮음
6,화재/경보,Alarm,0.000124,5,낮음
7,아기 울음,"Crying, sobbing",0.000115,2,낮음
8,초인종,Doorbell,0.000009,1,낮음


In [22]:
# Cell 9. Windowed inference
window_df = runner.predict_windows(
    AUDIO_PATH,
    window_sec=3.0,
    hop_sec=1.5,
    top_k=5,
)

display(window_df.head(50))

/content/EfficientAT/models/preprocess.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


,window_id,start_sec,end_sec,rank,audioset_label,probability
0,0,0.0,3.000000,1,Sanding,0.940881
1,0,0.0,3.000000,2,Sawing,0.912855
2,0,0.0,3.000000,3,Bicycle bell,0.910600
3,0,0.0,3.000000,4,Hammer,0.899597
4,0,0.0,3.000000,5,Typewriter,0.845920
5,1,1.5,4.500000,1,Sanding,0.987301
6,1,1.5,4.500000,2,Sawing,0.983326
7,1,1.5,4.500000,3,Hammer,0.975132
8,1,1.5,4.500000,4,Bicycle bell,0.954875
9,1,1.5,4.500000,5,Jackhammer,0.949028


In [23]:
# Cell 10. 결과 저장
out_csv = "efficientat_window_result.csv"
window_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("Saved:", out_csv)

Saved: efficientat_window_result.csv


In [24]:
from google.colab import files
files.download("efficientat_window_result.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
# 1) 공식 샘플로 동작 확인
AUDIO_PATH = "resources/metro_station-paris.wav"
runner = EfficientATRunner(model_name="mn10_as")
top_df, probs = runner.predict_file(AUDIO_PATH, top_k=10, return_probs=True)
display(top_df)

# 2) 관심 라벨 필터 확인
display(target_scores_from_probs(probs))

# 3) window 추론 확인
window_df = runner.predict_windows(AUDIO_PATH, window_sec=3.0, hop_sec=1.5, top_k=5)
display(window_df.head(30))

Loaded EfficientAT: mn10_as
Device: cpu


/usr/local/lib/python3.12/dist-packages/torchvision/ops/misc.py:121: UserWarning: Don't use ConvNormActivation directly, please use Conv2dNormActivation and Conv3dNormActivation instead.
  warnings.warn(
/content/EfficientAT/models/preprocess.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


,rank,audioset_label,probability,audio_path,inference_sec
0,1,Train,0.810543,resources/metro_station-paris.wav,0.095408
1,2,Rail transport,0.647846,resources/metro_station-paris.wav,0.095408
2,3,"Railroad car, train wagon",0.629242,resources/metro_station-paris.wav,0.095408
3,4,"Subway, metro, underground",0.550005,resources/metro_station-paris.wav,0.095408
4,5,Vehicle,0.327353,resources/metro_station-paris.wav,0.095408
5,6,Clickety-clack,0.177040,resources/metro_station-paris.wav,0.095408
6,7,Speech,0.062445,resources/metro_station-paris.wav,0.095408
7,8,"Outside, urban or manmade",0.029448,resources/metro_station-paris.wav,0.095408
8,9,Music,0.025280,resources/metro_station-paris.wav,0.095408
9,10,Train wheels squealing,0.021179,resources/metro_station-paris.wav,0.095408


,target_name_ko,best_audioset_label,probability,matched_label_count
0,차량 경적,Train horn,0.007879,5
1,물/누수 후보,"Boat, Water vehicle",0.002185,7
2,사이렌,Siren,0.000336,6
3,화재/경보,Fire alarm,0.000250,5
4,개 짖음,Bark,0.000214,1
5,비명/고함,Children shouting,0.000123,4
6,전화벨,Ringtone,0.000100,3
7,아기 울음,"Baby cry, infant cry",0.000089,2
8,초인종,Doorbell,0.000071,1


/content/EfficientAT/models/preprocess.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


,window_id,start_sec,end_sec,rank,audioset_label,probability
0,0,0.0,3.0,1,Hammer,0.948633
1,0,0.0,3.0,2,Sawing,0.776435
2,0,0.0,3.0,3,Cowbell,0.753315
3,0,0.0,3.0,4,Coin (dropping),0.731792
4,0,0.0,3.0,5,Bicycle bell,0.708716
5,1,1.5,4.5,1,Hammer,0.561378
6,1,1.5,4.5,2,"Cattle, bovinae",0.492542
7,1,1.5,4.5,3,Grunt,0.491068
8,1,1.5,4.5,4,Bicycle bell,0.488917
9,1,1.5,4.5,5,"Roaring cats (lions, tigers)",0.451426


In [26]:
# Cell FT-0. Fine-tuning dataset zip 업로드 및 train/valid 준비

import os
import zipfile
import shutil
import random
import json
from pathlib import Path
from collections import Counter

from google.colab import files as colab_files

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

RAW_DIR = Path("/content/ea_finetune_raw")
SPLIT_DIR = Path("/content/ea_finetune_split")

AUDIO_EXTS = {
    ".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac"
}

print("Fine-tuning dataset zip 파일을 업로드하세요.")
uploaded = colab_files.upload()
zip_name = next(iter(uploaded.keys()))

shutil.rmtree(RAW_DIR, ignore_errors=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(RAW_DIR)

def get_effective_root(root: Path) -> Path:
    items = [
        p for p in root.iterdir()
        if not p.name.startswith("__MACOSX") and not p.name.startswith(".")
    ]
    dirs = [p for p in items if p.is_dir()]
    files = [p for p in items if p.is_file()]
    if len(dirs) == 1 and len(files) == 0:
        return dirs[0]
    return root

def list_audio_files(path: Path):
    return sorted([
        p for p in path.rglob("*")
        if p.suffix.lower() in AUDIO_EXTS and not p.name.startswith("._")
    ])

DATA_ROOT = get_effective_root(RAW_DIR)

# zip 안에 train/valid가 이미 있으면 그대로 사용
if (DATA_ROOT / "train").exists() and (DATA_ROOT / "valid").exists():
    TRAIN_DIR = DATA_ROOT / "train"
    VALID_DIR = DATA_ROOT / "valid"
    print("Detected existing train/valid split.")

else:
    # train만 있으면 train 내부 class 폴더를 raw class 폴더로 간주
    if (DATA_ROOT / "train").exists() and not (DATA_ROOT / "valid").exists():
        DATA_ROOT = DATA_ROOT / "train"

    print("No valid split found. Creating train/valid split automatically.")

    shutil.rmtree(SPLIT_DIR, ignore_errors=True)
    TRAIN_DIR = SPLIT_DIR / "train"
    VALID_DIR = SPLIT_DIR / "valid"

    class_dirs = [
        p for p in DATA_ROOT.iterdir()
        if p.is_dir() and not p.name.startswith(".") and not p.name.startswith("__")
    ]

    assert len(class_dirs) > 0, "클래스 폴더를 찾지 못했습니다. zip 구조를 확인하세요."

    for class_dir in sorted(class_dirs):
        class_name = class_dir.name
        audio_files = list_audio_files(class_dir)

        if len(audio_files) == 0:
            print(f"Skip empty class folder: {class_name}")
            continue

        random.shuffle(audio_files)

        if len(audio_files) == 1:
            # 샘플이 1개뿐이면 데모용으로 train/valid 둘 다에 복사
            train_files = audio_files
            valid_files = audio_files
        else:
            n_valid = max(1, int(round(len(audio_files) * 0.2)))
            n_valid = min(n_valid, len(audio_files) - 1)
            valid_files = audio_files[:n_valid]
            train_files = audio_files[n_valid:]

        for split_name, split_files in [("train", train_files), ("valid", valid_files)]:
            out_class_dir = SPLIT_DIR / split_name / class_name
            out_class_dir.mkdir(parents=True, exist_ok=True)

            for src in split_files:
                dst = out_class_dir / src.name
                # 이름 중복 방지
                if dst.exists():
                    dst = out_class_dir / f"{src.stem}_{random.randint(10000, 99999)}{src.suffix}"
                shutil.copy2(src, dst)

CLASS_NAMES = sorted([
    p.name for p in TRAIN_DIR.iterdir()
    if p.is_dir() and len(list_audio_files(p)) > 0
])

assert len(CLASS_NAMES) >= 2, "파인튜닝하려면 최소 2개 클래스가 필요합니다. 예: car_traffic, construction_tool, background"

class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

print("Classes:", CLASS_NAMES)
print("class_to_idx:", class_to_idx)

def count_by_class(root: Path):
    rows = []
    for cname in CLASS_NAMES:
        rows.append({
            "class": cname,
            "count": len(list_audio_files(root / cname))
        })
    return rows

print("Train counts:", count_by_class(TRAIN_DIR))
print("Valid counts:", count_by_class(VALID_DIR))

Fine-tuning dataset zip 파일을 업로드하세요.


Saving efficientat_ft_starter_dataset.zip to efficientat_ft_starter_dataset.zip
Detected existing train/valid split.
Classes: ['background', 'car_traffic', 'construction_tool', 'horn', 'siren']
class_to_idx: {'background': 0, 'car_traffic': 1, 'construction_tool': 2, 'horn': 3, 'siren': 4}
Train counts: [{'class': 'background', 'count': 30}, {'class': 'car_traffic', 'count': 30}, {'class': 'construction_tool', 'count': 30}, {'class': 'horn', 'count': 30}, {'class': 'siren', 'count': 30}]
Valid counts: [{'class': 'background', 'count': 8}, {'class': 'car_traffic', 'count': 8}, {'class': 'construction_tool', 'count': 8}, {'class': 'horn', 'count': 8}, {'class': 'siren', 'count': 8}]


In [27]:
# Cell FT-1. Audio dataset 정의

import os
import sys
import math
import time
import contextlib
import io
import numpy as np
import pandas as pd
import librosa
import torch

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SAMPLE_RATE = 32000
CLIP_SEC = 3.0
BATCH_SIZE = 16

class AudioFolderDataset(Dataset):
    def __init__(
        self,
        root_dir: Path,
        class_names: list[str],
        sample_rate: int = SAMPLE_RATE,
        clip_sec: float = CLIP_SEC,
        train: bool = True,
    ):
        self.root_dir = Path(root_dir)
        self.class_names = class_names
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.sample_rate = sample_rate
        self.clip_sec = clip_sec
        self.train = train
        self.target_len = int(sample_rate * clip_sec)

        self.samples = []

        for cname in class_names:
            cdir = self.root_dir / cname
            for path in list_audio_files(cdir):
                self.samples.append((str(path), self.class_to_idx[cname]))

        if len(self.samples) == 0:
            raise ValueError(f"No audio files found in {self.root_dir}")

    def __len__(self):
        return len(self.samples)

    def _crop_or_pad(self, wav: np.ndarray) -> np.ndarray:
        n = len(wav)

        if n == 0:
            wav = np.zeros(self.target_len, dtype=np.float32)
            return wav

        if n < self.target_len:
            pad = self.target_len - n
            wav = np.pad(wav, (0, pad))
            return wav.astype(np.float32)

        if n > self.target_len:
            if self.train:
                start = random.randint(0, n - self.target_len)
            else:
                start = (n - self.target_len) // 2
            wav = wav[start:start + self.target_len]

        return wav.astype(np.float32)

    def _augment(self, wav: np.ndarray) -> np.ndarray:
        if not self.train:
            return wav

        # gain jitter
        if random.random() < 0.8:
            gain_db = random.uniform(-6.0, 6.0)
            wav = wav * (10.0 ** (gain_db / 20.0))

        # light gaussian noise
        if random.random() < 0.3:
            noise_level = random.uniform(0.001, 0.006)
            wav = wav + np.random.randn(len(wav)).astype(np.float32) * noise_level

        # time shift
        if random.random() < 0.3:
            shift = random.randint(-int(0.15 * self.sample_rate), int(0.15 * self.sample_rate))
            wav = np.roll(wav, shift)

        return np.clip(wav, -1.0, 1.0).astype(np.float32)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        wav, _ = librosa.load(
            path,
            sr=self.sample_rate,
            mono=True,
        )

        wav = wav.astype(np.float32)
        wav = self._crop_or_pad(wav)
        wav = self._augment(wav)

        return torch.from_numpy(wav), torch.tensor(label, dtype=torch.long), path


train_ds = AudioFolderDataset(
    TRAIN_DIR,
    CLASS_NAMES,
    sample_rate=SAMPLE_RATE,
    clip_sec=CLIP_SEC,
    train=True,
)

valid_ds = AudioFolderDataset(
    VALID_DIR,
    CLASS_NAMES,
    sample_rate=SAMPLE_RATE,
    clip_sec=CLIP_SEC,
    train=False,
)

batch_size = min(BATCH_SIZE, max(2, len(train_ds)))

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    drop_last=False,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

print("Train samples:", len(train_ds))
print("Valid samples:", len(valid_ds))
print("Batch size:", batch_size)

Train samples: 150
Valid samples: 40
Batch size: 16


In [28]:
# Cell FT-2. EfficientAT model 구성

import os
import sys
import torch
import torch.nn.functional as F

os.chdir("/content/EfficientAT")
sys.path.insert(0, "/content/EfficientAT")

from models.mn.model import get_model as get_mn
from models.preprocess import AugmentMelSTFT
from helpers.utils import NAME_TO_WIDTH

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_MODEL = "mn10_as"  # 더 가볍게 하려면 "mn04_as" 또는 "mn05_as"
HEAD_TYPE = "mlp"

N_MELS = 128
WIN_LENGTH = 800
HOP_SIZE = 320

# num_classes를 프로젝트 클래스 수로 바꿔서 새 classifier를 붙임
with contextlib.redirect_stdout(io.StringIO()):
    model = get_mn(
        num_classes=len(CLASS_NAMES),
        width_mult=NAME_TO_WIDTH(BASE_MODEL),
        pretrained_name=BASE_MODEL,
        head_type=HEAD_TYPE,
    )

model = model.to(DEVICE)

mel = AugmentMelSTFT(
    n_mels=N_MELS,
    sr=SAMPLE_RATE,
    win_length=WIN_LENGTH,
    hopsize=HOP_SIZE,
).to(DEVICE)

# 1단계: backbone freeze, classifier만 학습
for p in model.features.parameters():
    p.requires_grad = False

for p in model.classifier.parameters():
    p.requires_grad = True

def count_trainable_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("Device:", DEVICE)
print("Base model:", BASE_MODEL)
print("Project classes:", CLASS_NAMES)
print("Trainable params:", count_trainable_params(model))

Device: cpu
Base model: mn10_as
Project classes: ['background', 'car_traffic', 'construction_tool', 'horn', 'siren']
Trainable params: 1236485


/usr/local/lib/python3.12/dist-packages/torchvision/ops/misc.py:121: UserWarning: Don't use ConvNormActivation directly, please use Conv2dNormActivation and Conv3dNormActivation instead.
  warnings.warn(


In [29]:
# Cell FT-3. Training / validation 함수

def wav_batch_to_spec(wavs: torch.Tensor) -> torch.Tensor:
    """
    wavs: [B, samples]
    return: [B, 1, n_mels, time]
    """
    specs = mel(wavs)

    if specs.dim() == 3:
        specs = specs.unsqueeze(1)
    elif specs.dim() == 4:
        pass
    else:
        raise RuntimeError(f"Unexpected spec shape: {specs.shape}")

    return specs


def set_model_mode(train: bool):
    if train:
        model.train()

        # requires_grad=False인 frozen feature block의 BatchNorm 통계 업데이트 방지
        for layer in model.features:
            has_trainable = any(p.requires_grad for p in layer.parameters())
            if not has_trainable:
                layer.eval()
    else:
        model.eval()

    # spectrogram front-end는 고정적으로 사용
    mel.eval()


criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3,
    weight_decay=1e-4,
)

EPOCHS = 12
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)

def run_epoch(loader, train: bool):
    set_model_mode(train)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    all_targets = []
    all_preds = []

    pbar = tqdm(loader, leave=False)

    for wavs, targets, paths in pbar:
        wavs = wavs.to(DEVICE)
        targets = targets.to(DEVICE)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            specs = wav_batch_to_spec(wavs)
            logits, features = model(specs)
            loss = criterion(logits, targets)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        preds = logits.argmax(dim=1)

        total_loss += loss.item() * wavs.size(0)
        total_correct += (preds == targets).sum().item()
        total_count += wavs.size(0)

        all_targets.extend(targets.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        pbar.set_description(
            f"{'train' if train else 'valid'} loss={loss.item():.4f}"
        )

    avg_loss = total_loss / max(1, total_count)
    acc = total_correct / max(1, total_count)

    return avg_loss, acc, np.array(all_targets), np.array(all_preds)

In [30]:
# Cell FT-4. Head-only fine-tuning 실행

BEST_PATH = "/content/efficientat_finetuned_best.pt"

best_valid_acc = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, train=True)
    valid_loss, valid_acc, y_true, y_pred = run_epoch(valid_loader, train=False)

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
        "lr": scheduler.get_last_lr()[0],
    }
    history.append(row)

    print(
        f"[Epoch {epoch:02d}/{EPOCHS}] "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f}, "
        f"valid_loss={valid_loss:.4f}, valid_acc={valid_acc:.3f}"
    )

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc

        torch.save(
            {
                "base_model": BASE_MODEL,
                "head_type": HEAD_TYPE,
                "class_names": CLASS_NAMES,
                "class_to_idx": class_to_idx,
                "sample_rate": SAMPLE_RATE,
                "clip_sec": CLIP_SEC,
                "n_mels": N_MELS,
                "win_length": WIN_LENGTH,
                "hop_size": HOP_SIZE,
                "model_state_dict": model.state_dict(),
                "valid_acc": valid_acc,
                "epoch": epoch,
            },
            BEST_PATH,
        )

        print("Saved best checkpoint:", BEST_PATH)

history_df = pd.DataFrame(history)
display(history_df)

print("Best valid acc:", best_valid_acc)

  0%|          | 0/10 [00:00<?, ?it/s]

/content/EfficientAT/models/preprocess.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


  0%|          | 0/3 [00:00<?, ?it/s]

[Epoch 01/12] train_loss=1.1757, train_acc=0.680, valid_loss=0.6854, valid_acc=1.000
Saved best checkpoint: /content/efficientat_finetuned_best.pt


  0%|          | 0/10 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[Epoch 12/12] train_loss=0.2520, train_acc=0.987, valid_loss=0.2394, valid_acc=1.000


,epoch,train_loss,train_acc,valid_loss,valid_acc,lr
0,1,1.175729,0.680000,0.685409,1.00,0.000983
1,2,0.509089,0.973333,0.374345,0.95,0.000933
2,3,0.343536,0.973333,0.274191,1.00,0.000854
3,4,0.293247,0.993333,0.260197,1.00,0.000750
4,5,0.274647,0.993333,0.259122,1.00,0.000629
5,6,0.265609,0.993333,0.248171,1.00,0.000500
6,7,0.269936,0.986667,0.244899,1.00,0.000371
7,8,0.254521,0.993333,0.248162,1.00,0.000250
8,9,0.254510,0.993333,0.237049,1.00,0.000146
9,10,0.255761,0.993333,0.236384,1.00,0.000067


Best valid acc: 1.0


In [31]:
# Cell FT-5. Confusion matrix / classification report

from sklearn.metrics import classification_report, confusion_matrix

# best checkpoint 다시 로드
ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)

valid_loss, valid_acc, y_true, y_pred = run_epoch(valid_loader, train=False)

print("Valid loss:", valid_loss)
print("Valid acc:", valid_acc)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{c}" for c in CLASS_NAMES],
    columns=[f"pred_{c}" for c in CLASS_NAMES],
)

display(cm_df)

  0%|          | 0/3 [00:00<?, ?it/s]

/content/EfficientAT/models/preprocess.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


Valid loss: 0.6854088008403778
Valid acc: 1.0
                   precision    recall  f1-score   support

       background      1.000     1.000     1.000         8
      car_traffic      1.000     1.000     1.000         8
construction_tool      1.000     1.000     1.000         8
             horn      1.000     1.000     1.000         8
            siren      1.000     1.000     1.000         8

         accuracy                          1.000        40
        macro avg      1.000     1.000     1.000        40
     weighted avg      1.000     1.000     1.000        40



,pred_background,pred_car_traffic,pred_construction_tool,pred_horn,pred_siren
true_background,8,0,0,0,0
true_car_traffic,0,8,0,0,0
true_construction_tool,0,0,8,0,0
true_horn,0,0,0,8,0
true_siren,0,0,0,0,8


In [32]:
# Cell FT-6. Optional: last feature blocks까지 부분 unfreeze

# 마지막 feature block 몇 개만 unfreeze
N_UNFREEZE_BLOCKS = 4

for layer in model.features[-N_UNFREEZE_BLOCKS:]:
    for p in layer.parameters():
        p.requires_grad = True

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=1e-4,
)

EPOCHS_STAGE2 = 5

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS_STAGE2,
)

print("Trainable params after unfreeze:", count_trainable_params(model))

for epoch in range(1, EPOCHS_STAGE2 + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, train=True)
    valid_loss, valid_acc, y_true, y_pred = run_epoch(valid_loader, train=False)

    scheduler.step()

    print(
        f"[Stage2 Epoch {epoch:02d}/{EPOCHS_STAGE2}] "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f}, "
        f"valid_loss={valid_loss:.4f}, valid_acc={valid_acc:.3f}"
    )

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc

        torch.save(
            {
                "base_model": BASE_MODEL,
                "head_type": HEAD_TYPE,
                "class_names": CLASS_NAMES,
                "class_to_idx": class_to_idx,
                "sample_rate": SAMPLE_RATE,
                "clip_sec": CLIP_SEC,
                "n_mels": N_MELS,
                "win_length": WIN_LENGTH,
                "hop_size": HOP_SIZE,
                "model_state_dict": model.state_dict(),
                "valid_acc": valid_acc,
                "epoch": f"stage2_{epoch}",
            },
            BEST_PATH,
        )

        print("Saved improved checkpoint:", BEST_PATH)

print("Best valid acc after stage2:", best_valid_acc)

Trainable params after unfreeze: 3415949


  0%|          | 0/10 [00:00<?, ?it/s]

/content/EfficientAT/models/preprocess.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


  0%|          | 0/3 [00:00<?, ?it/s]

[Stage2 Epoch 01/5] train_loss=1.4247, train_acc=0.513, valid_loss=0.6281, valid_acc=1.000


...

  0%|          | 0/3 [00:00<?, ?it/s]

[Stage2 Epoch 04/5] train_loss=0.9256, train_acc=0.980, valid_loss=0.5728, valid_acc=1.000


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[Stage2 Epoch 05/5] train_loss=0.8862, train_acc=1.000, valid_loss=0.5789, valid_acc=1.000
Best valid acc after stage2: 1.0


In [33]:
# Cell FT-7. Fine-tuned EfficientAT window inference

import math

def safe_torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def load_finetuned_efficientat(ckpt_path=BEST_PATH):
    ckpt = safe_torch_load(ckpt_path, map_location=DEVICE)

    class_names = ckpt["class_names"]
    base_model = ckpt.get("base_model", BASE_MODEL)
    head_type = ckpt.get("head_type", "mlp")

    with contextlib.redirect_stdout(io.StringIO()):
        ft_model = get_mn(
            num_classes=len(class_names),
            width_mult=NAME_TO_WIDTH(base_model),
            pretrained_name=None,
            head_type=head_type,
        )

    ft_model.load_state_dict(ckpt["model_state_dict"])
    ft_model.to(DEVICE)
    ft_model.eval()

    ft_mel = AugmentMelSTFT(
        n_mels=ckpt.get("n_mels", N_MELS),
        sr=ckpt.get("sample_rate", SAMPLE_RATE),
        win_length=ckpt.get("win_length", WIN_LENGTH),
        hopsize=ckpt.get("hop_size", HOP_SIZE),
    ).to(DEVICE)
    ft_mel.eval()

    return ft_model, ft_mel, class_names, ckpt

ft_model, ft_mel, ft_class_names, ft_ckpt = load_finetuned_efficientat(BEST_PATH)

print("Loaded fine-tuned model")
print("Classes:", ft_class_names)
print("Checkpoint valid_acc:", ft_ckpt.get("valid_acc"))

def rms_dbfs(wav: np.ndarray):
    rms = np.sqrt(np.mean(np.square(wav)) + 1e-12)
    return float(20.0 * np.log10(rms + 1e-12))

def ft_wav_to_spec(wavs: torch.Tensor) -> torch.Tensor:
    specs = ft_mel(wavs)

    if specs.dim() == 3:
        specs = specs.unsqueeze(1)
    elif specs.dim() == 4:
        pass
    else:
        raise RuntimeError(f"Unexpected spec shape: {specs.shape}")

    return specs

def predict_ft_segment(wav_np: np.ndarray):
    wav_tensor = torch.from_numpy(wav_np[None, :]).float().to(DEVICE)

    with torch.inference_mode():
        specs = ft_wav_to_spec(wav_tensor)
        logits, features = ft_model(specs)
        probs = torch.softmax(logits.float(), dim=1).squeeze(0).cpu().numpy()

    return probs

# 프로젝트용 alert mapping. 폴더명을 바꿨으면 여기도 맞춰 수정.
ALERT_MAP = {
    "horn": "위험",
    "siren": "위험",
    "scream": "위험",
    "fire_alarm": "위험",

    "car_traffic": "주의",
    "construction_tool": "주의",

    "doorbell": "정보",
    "baby_cry": "정보",
    "background": "알림없음",
}

def predict_ft_windows(
    audio_path: str,
    window_sec: float = 3.0,
    hop_sec: float = 1.5,
    top_k: int = 3,
    silence_dbfs: float = -45.0,
):
    wav, _ = librosa.load(
        audio_path,
        sr=SAMPLE_RATE,
        mono=True,
    )

    wav = wav.astype(np.float32)

    win = int(window_sec * SAMPLE_RATE)
    hop = int(hop_sec * SAMPLE_RATE)
    n = len(wav)

    if n == 0:
        raise ValueError("Audio file is empty.")

    n_windows = max(1, math.ceil(max(0, n - win) / hop) + 1)

    records = []

    for window_id in range(n_windows):
        s = window_id * hop
        e = s + win

        raw_seg = wav[s:min(e, n)]

        if len(raw_seg) < win:
            seg = np.pad(raw_seg, (0, win - len(raw_seg)))
        else:
            seg = raw_seg

        db = rms_dbfs(raw_seg)
        probs = predict_ft_segment(seg)

        top_idx = np.argsort(probs)[::-1][:top_k]

        for rank, idx in enumerate(top_idx, start=1):
            label = ft_class_names[idx]
            prob = float(probs[idx])

            if db < silence_dbfs:
                alert = "no_alert_low_energy"
            else:
                alert = ALERT_MAP.get(label, "확인필요")

            records.append({
                "window_id": window_id,
                "start_sec": s / SAMPLE_RATE,
                "end_sec": min(e, n) / SAMPLE_RATE,
                "rms_dbfs": db,
                "is_low_energy": db < silence_dbfs,
                "rank": rank,
                "label": label,
                "probability": prob,
                "alert_level": alert,
            })

    return pd.DataFrame(records)

Loaded fine-tuned model
Classes: ['background', 'car_traffic', 'construction_tool', 'horn', 'siren']
Checkpoint valid_acc: 1.0


/usr/local/lib/python3.12/dist-packages/torchvision/ops/misc.py:121: UserWarning: Don't use ConvNormActivation directly, please use Conv2dNormActivation and Conv3dNormActivation instead.
  warnings.warn(


In [34]:
# Cell FT-8. 테스트 MP3 업로드 후 fine-tuned 결과 CSV 생성

try:
    TEST_AUDIO = AUDIO_PATH
    print("Using existing AUDIO_PATH:", TEST_AUDIO)
except NameError:
    print("테스트할 mp3/wav 파일을 업로드하세요.")
    uploaded_test = colab_files.upload()
    TEST_AUDIO = next(iter(uploaded_test.keys()))

ft_window_df = predict_ft_windows(
    TEST_AUDIO,
    window_sec=3.0,
    hop_sec=1.5,
    top_k=3,
    silence_dbfs=-45.0,
)

display(ft_window_df.head(80))

out_csv = "efficientat_finetuned_window_result.csv"
ft_window_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("Saved:", out_csv)
colab_files.download(out_csv)

Using existing AUDIO_PATH: resources/metro_station-paris.wav


,window_id,start_sec,end_sec,rms_dbfs,is_low_energy,rank,label,probability,alert_level
0,0,0.0,3.0,-26.889746,False,1,car_traffic,0.321291,주의
1,0,0.0,3.0,-26.889746,False,2,construction_tool,0.251077,주의
2,0,0.0,3.0,-26.889746,False,3,background,0.162356,알림없음
3,1,1.5,4.5,-20.420897,False,1,car_traffic,0.346126,주의
4,1,1.5,4.5,-20.420897,False,2,construction_tool,0.238577,주의
5,1,1.5,4.5,-20.420897,False,3,horn,0.154527,위험
6,2,3.0,6.0,-14.374388,False,1,car_traffic,0.413421,주의
7,2,3.0,6.0,-14.374388,False,2,construction_tool,0.220704,주의
8,2,3.0,6.0,-14.374388,False,3,horn,0.138118,위험
9,3,4.5,7.5,-12.622446,False,1,car_traffic,0.324212,주의


Saved: efficientat_finetuned_window_result.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
# Cell FT-9. fine-tuned checkpoint 다운로드

colab_files.download(BEST_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>